# EQO notebook: NWQ-Sim CPU simulation

Submit a bounded OpenQASM 2.0 circuit to EQO's published NWQ-Sim CPU state-vector operation. The notebook is only a control-plane client: it does not import NWQ-Sim, pull an image, or start a container.

Start EQO Local first, then install the notebook extra:

```bash
python -m pip install -e ".[jupyter]"
eqo local up
```

In [ ]:
import os

from eqo import EQOClient, render_artifact, render_run

eqo = EQOClient.connect(os.environ.get("EQO_ENDPOINT", "http://127.0.0.1:8080"))
eqo.health()
published = {item["id"]: item for item in eqo.workflows.list()}
workflow = published.get("nwqsim-bell-simulation")
if workflow is None:
    raise RuntimeError("nwqsim-bell-simulation is not published by this EQO profile. Restart EQO Local.")
workflow

## Create a typed OpenQASM input

The admitted CPU operation accepts one bounded OpenQASM 2.0 circuit. This Bell circuit is deliberately small; the operation's declared limits remain enforced by EQO.

In [ ]:
bell_qasm = """OPENQASM 2.0;
include \"qelib1.inc\";
qreg q[2];
creg c[2];
h q[0];
cx q[0],q[1];
measure q -> c;
"""

circuit = eqo.artifacts.create_input(
    "qhpc.quantum-circuit@1",
    bell_qasm,
    name="bell.qasm",
    labels={"example": "nwqsim-cpu"},
)
render_artifact(circuit)

## Run NWQ-Sim through EQO

The workflow selects the pinned Linux/AMD64 NWQ-Sim CPU image and records the run, artifact checksums, seed, and measurement counts.

In [ ]:
run = eqo.workflows.submit(
    workflow["id"],
    workflow["version"],
    inputs={"circuit": circuit.id},
)
render_run(run)
completed = run.wait(timeout=300)
if completed.state != "succeeded":
    raise RuntimeError(f"NWQ-Sim run ended in {completed.state}; inspect render_run(completed).")
render_run(completed)
render_artifact(completed.artifacts.by_type("qhpc.nwqsim-measurement-counts@1"))